In [1]:
%load_ext autoreload
%autoreload 2

Formula 1 (F1) is the highest class of single-seater auto racing sanctioned by the Fédération Internationale de l'Automobile (FIA). The series has its roots in the European Championship of Grand Prix motor racing of the 1920s and 1930s. The "formula" in the name refers to a set of rules that all participants' cars must meet. The F1 World Championship, which consists of a series of races known as Grands Prix, was first established in 1950. Since then, the sport has evolved and grown in popularity, with F1 cars becoming faster, more advanced, and more sophisticated with each passing year. Today, F1 is considered the pinnacle of motor racing and is followed by millions of fans around the world.



In the mid 1950's the cars were simpler than they were today, and the formula required an engine which has a capacity of 2.5L naturally aspirated, or 0.75L forced induction. Our car uses a 2.5L naturally aspirated engine

### Single responsibility principle

In [2]:
from app.f1_cars.f1_car_v1 import F1Car_v1

Throughout the season, F1 engineering teams continuously tests & improves different parts of the car. Each time this happens, the car needs to be tested to make sure it still runs. With the current implementation of our F1 car there is a glaring issue the engineering team isn't able to test the engine, or any of it's parts in isolation. There are two choices for testing the F1 car:

1. get the f1 car back into the factory for testing

In [3]:
def test_f1_car_engine_by_transporting_it_back_to_the_factory():
    f1_car = transport_f1_car_to_factory()
    f1_car.start_engine()
    assert f1_car.engine.has_started == True


def transport_f1_car_to_factory():
    print("logic to transport f1 car to factory")
    return F1Car_v1()

In [4]:
test_f1_car_engine_by_transporting_it_back_to_the_factory()

logic to transport f1 car to factory
engine has started


2. construct a new f1 car 

In [5]:
def test_f1_car_engine_by_constructing_a_new_one():
    f1_car = F1Car_v1()
    f1_car.start_engine()
    assert f1_car.engine.has_started == True

In [6]:
test_f1_car_engine_by_constructing_a_new_one()

engine has started


Even though at the current day and age, F1 teams are [logistical experts](https://www.youtube.com/watch?v=6OLVFa8YRfM) in the past having a mobile R&D facility was out of reach. Given that teams don't have a factory in all the places they're racing this would put the teams that can't the different parts of the car in isolation a problem! 

To resolve this issue, we could delegate the creation of the engine, chassis, and wheels to the factory so the car does not need to know how they're implemented

In [7]:
from app.chassis import ChassisFactory
from app.engine import EngineFactory
from app.fuel_tank import FuelTankFactory
from app.wheels import WheelsFactory

Then we could update the class like so:

In [8]:
from app.f1_cars.f1_car_v2 import F1Car_v2

Then the engine can be easily tested!

In [9]:
def test_f1_car_engine():
    engine = EngineFactory.create_engine_1950s_2_5L_naturally_aspirated_v1()
    engine.start()
    print("check that the engine has started")

In [10]:
test_f1_car_engine()

engine has started
check that the engine has started


By removing the construction of the parts of the `F1Car` from it's responsibilities, we have in effect applied the Single Responsibility Principle. Since the car is not responsible for building the engine, if there is an issue during the construction of the engine it can be isolated to the factory, moreover, the engine is able to be tested independently of the car! 

### Open closed principle

As a result of the engines being able to be tested independently of the car. The engineers were able to research and develop new iterations of the engine and they've created a new engine `Engine_v2`!

In [11]:
engine = EngineFactory.create_engine_1950s_2_5L_naturally_aspirated_v2()

Now this new engine needs to be installed into the car. Though, can you see the problem? Although we've solved the problem of being able to test the engine in isolation, the car (the code) can't be reused, and it needs to be opened up for modifications and rebuilt. In particular, to use the new engine, the line:

`self.engine = EngineFactory_v1.create_engine_v1` would have to be changed to `self.engine = EngineFactory_v2.create_engine_v2` something like so:

In [12]:
from app.f1_cars.f1_car_v3 import F1Car_v3

This is not ideal as the other parts of the car, the chassis and the wheels are still perfectly fine, but they can't be reused. Moreover, changing the whole car means more testing and more time taken! To resolve this issue, we could change the way the car is modeled by accepting the parts needed by the car in the constructor instead:

In [13]:
from app.f1_cars.f1_car_v4 import F1Car_v4

Then to represent the car with the new engine we first update the factory to create the new engine:

change:

In [14]:
f1_car = F1Car_v3()

to

In [15]:
f1_car = F1Car_v4(
    EngineFactory.create_engine_1950s_2_5L_naturally_aspirated_v2(),
    ChassisFactory.create_chassis_v1(),
    WheelsFactory.create_wheels_v1(),
    FuelTankFactory.create_fuel_tank_v1(),
)

By modifying the code to take in the components that it needs, we have changed the car such that it's closed for modifications as regardless of the car part being used, the car does not need to change, and also it is open for extension (partially, but we will get to that later) as it will be able to take in a new type of engine without having it change the code.

### Dependency Inversion Principle

By the mid 1980s, all the cars were using BMW's straight-4 turbo, the M12/13 engine which was a 1.5L engine with turbocharger (picture). As a result of this, the way the engine had to be started was different so instead of calling `start` to start the engine, it had to be called `start_with_turbocharger`  but they didn't communicate this to the team! The new engine was shipped and installed into the car:

In [24]:
f1_car = F1Car_v4(
    EngineFactory.create_engine_1980s_1_5L_turbocharged_v1(),
    ChassisFactory.create_chassis_v1(),
    WheelsFactory.create_wheels_v1(),
    FuelTankFactory.create_fuel_tank_v1(),
)

When it came to start the car, an error appeared!

In [25]:
from app.drivers.driver_v1 import Driver_v1

In [26]:
driver = Driver_v1()
driver.start_car(f1_car)

AttributeError: 'Engine1980s_1_5L_TurboCharged_v1' object has no attribute 'start'

Since the way the to start the engine has changed (the interface) the f1 car could not be started! This is a disaster! As a workaround, the team made a hacky fix to update the car such that it changes the way an engine should be started depending on the type of engine it is receiving like so: 

In [27]:
from app.f1_cars.f1_car_v4_patched import F1Car_v4_patched

In [28]:
f1_car = F1Car_v4_patched(
    EngineFactory.create_engine_1980s_1_5L_turbocharged_v1(),
    ChassisFactory.create_chassis_v1(),
    WheelsFactory.create_wheels_v1(),
    FuelTankFactory.create_fuel_tank_v1(),
)

In [29]:
driver = Driver_v1()
driver.start_car(f1_car)

engine has started with turbocharger


This works, but the same issue arises, if the engine being used by the car requires a different to start it, for example if a new version of the engine was created `Engine_v4` and had a different way to start the engine `start_v4` the car would have to be opened up for modifications again, which isn't ideal. To remedy the situation, the team created a specification which required all engines to be started in the same way:

In [30]:
from app.engine.interface import EngineInterface

This way, regardless of the modifications made to the engine, the way to start it would be the same, to put it into action we would make `EngineV3` a subclass of IEngine. This is a caveat in Python as there are no interface types so one of the ways to implement it is to use an abstract class with abstract methods:

In [31]:
from app.engine.engine_1980s_1_5L_turbocharged_v2 import (
    Engine1980s_1_5L_TurboCharged_v2,
)

Let's see what happens when we try to use it!

In [32]:
f1_car = F1Car_v4(
    EngineFactory.create_engine_1980s_1_5L_turbocharged_v2(),
    ChassisFactory.create_chassis_v1(),
    WheelsFactory.create_wheels_v1(),
    FuelTankFactory.create_fuel_tank_v1(),
)

TypeError: Can't instantiate abstract class Engine1980s_1_5L_TurboCharged_v2 with abstract method start

Here we see that the current engine does not conform to the specification that has been set forward by the team hence an error is being thrown

To remedy, we need to implement the `start` function

In [33]:
from app.engine.engine_1980s_1_5L_turbocharged_v3 import (
    Engine1980s_1_5L_TurboCharged_v3,
)

Now instead of the car depending on a class, it now depends on a interface. In other words, the dependencies of the car has been inverted. Now the car doesn't car about what kind of engine is being passed to it, as long as the engine implements the `start` method eveything should work the same way

In [34]:
f1_car = F1Car_v4(
    EngineFactory.create_engine_1980s_1_5L_turbocharged_v3(),
    ChassisFactory.create_chassis_v1(),
    WheelsFactory.create_wheels_v1(),
    FuelTankFactory.create_fuel_tank_v1(),
)

What we have done here is to depend on an abstraction instead of a concrete implementation. In this case, the concrete implementation is `EngineV3` whereas the abstraction is the `EngineSpecification` interface. This is an application of the dependency inversion principle.

In [35]:
driver = Driver_v1()
driver.start_car(f1_car)

engine has started with turbocharger


### Liskov Substitution Principle

The formal definition of the Liskov Substitution Principle is:
```
Liskov's notion of a behavioural subtype defines a notion of substitutability for objects; that is, if S is a subtype of T, then objects of type T in a program may be replaced with objects of type S without altering any of the desirable properties of that program.
```

In a nutshell, it means one can find and replace all implementation of a superclass with it's subclass and the code should still work the same way. To break this down Barbara Liskov (the creator of the Liskov Substitution Principle) has detailed three main rules:
1. The **signature rule**
2. The **properties rule**
3. The **methods rule**

#### Liskov Substitution Principle - signature rule

The **signature rule** which states:
1. functions in the superclass should be implemented by the subclass
2. exceptions thrown in the subclass should be a subset of the exceptions handled by the superclass

The 1980s also introduced telemetry to the car, as a result the car had to be upgrades. Learning from past mistakes, the team used this as a chance to create an interface for the `F1Car` so not only does the car depend on an abstraction of an engine, but the driver also depends on an abstraction of the car

In [1]:
from app.f1_cars.f1_car_with_telemetry_v1 import F1CarWithTelemetry_v1
from app.f1_cars.interface import F1CarInterface
from app.telemetry_system import TelemetrySystemFactory

As a result of this, the driver needed to learn how to start the telemetry

In [2]:
from app.drivers.driver_v2 import Driver_v2

Let's test out the car and see what telemetry gets recorded!

In [42]:
f1_car = f1_car = F1CarWithTelemetry_v1(
    EngineFactory.create_engine_1980s_1_5L_turbocharged_v3(),
    ChassisFactory.create_chassis_v1(),
    WheelsFactory.create_wheels_v1(),
    FuelTankFactory.create_fuel_tank_v1(),
    TelemetrySystemFactory.create_telemetry_system_v1(),
)

TypeError: Can't instantiate abstract class F1CarWithTelemetry_v1 with abstract method get_current_telemetry

Oops! Seems like the method `get_current_telemetry` wasn't implemented! Good thing we caught this with our interface

In [43]:
from app.f1_cars.f1_car_with_telemetry_v2 import F1CarWithTelemetry_v2

In [44]:
f1_car = f1_car = F1CarWithTelemetry_v2(
    EngineFactory.create_engine_1980s_1_5L_turbocharged_v3(),
    ChassisFactory.create_chassis_v1(),
    WheelsFactory.create_wheels_v1(),
    FuelTankFactory.create_fuel_tank_v1(),
    TelemetrySystemFactory.create_telemetry_system_v1(),
)
driver = Driver_v2()
driver.start_car(f1_car)

engine has started with turbocharger


In [30]:
driver.accelerate_car(f1_car, fuel_amount=5)

inject air into engine with better efficiency
inject fuel into engine with better efficiency


In [31]:
f1_car.get_all_telemetry()

AttributeError: 'F1CarWithTelemetry_v2' object has no attribute 'get_all_telemetry'

What why is it empty?? Turns out that the driver forgot to enable the telemetry good thing this was caught in testing, all goood the driver just needs to remember to enable the telemetry before the car starts, and if the telemetry isn't enabled then the car will thrown an error!

In [32]:
from app.f1_cars.f1_car_with_telemetry_v3 import F1CarWithTelemetry_v3

In [33]:
f1_car = f1_car = F1CarWithTelemetry_v3(
    EngineFactory.create_engine_1980s_1_5L_turbocharged_v3(),
    ChassisFactory.create_chassis_v1(),
    WheelsFactory.create_wheels_v1(),
    FuelTankFactory.create_fuel_tank_v1(),
    TelemetrySystemFactory.create_telemetry_system_v1(),
)
driver = Driver_v2()
driver.enable_car_telemetry(f1_car)
driver.start_car(f1_car)

engine has started with turbocharger


In [34]:
driver.accelerate_car(f1_car, fuel_amount=5)

inject air into engine with better efficiency
inject fuel into engine with better efficiency


In [35]:
f1_car.get_telemetry_logs()

[{'fuel': 100}]

Amazing everything works, let's call it a day and start on the track tomorrow!

In [36]:
f1_car = F1CarWithTelemetry_v3(
    EngineFactory.create_engine_1980s_1_5L_turbocharged_v3(),
    ChassisFactory.create_chassis_v1(),
    WheelsFactory.create_wheels_v1(),
    FuelTankFactory.create_fuel_tank_v1(),
    TelemetrySystemFactory.create_telemetry_system_v1(),
)

The driver settles in and start the engine

In [37]:
driver = Driver_v2()
driver.start_car(f1_car)

TelemetryNotEnabledError: The engine has started but the telemetry hasn't been enabled!

Oh no! The car can't start, and the driver can't seem to figure out what it is! How has this happened? The car had implemented all the interfaces, and the driver depended on the abstraction! 
No worries it can be fixed

In [38]:
from app.drivers.driver_v3 import Driver_v3

In [39]:
driver = Driver_v3()
f1_car = f1_car = F1CarWithTelemetry_v3(
    EngineFactory.create_engine_1980s_1_5L_turbocharged_v3(),
    ChassisFactory.create_chassis_v1(),
    WheelsFactory.create_wheels_v1(),
    FuelTankFactory.create_fuel_tank_v1(),
    TelemetrySystemFactory.create_telemetry_system_v1(),
)
driver.start_car(f1_car)

Car telemetry is not enabled. Enabling car telemetry...
engine has started with turbocharger


Even though an interface was defined, and all the methods were implemented the driver code still needed to be opened up for modification as the driver was unaware of new exceptions that were thrown! This is a violation of part one, out of three, of the Liskov substitution principle called the **signature rule** which states:

1. functions in the superclass should be implemented by the subclass, this was enforced by the F1CarInterface as we saw when the car didn't implement the `get_all_telemetry` function
2. exceptions thrown in the subclass should be a subset of the exceptions handled by the superclass, here the `TelemetryNotEnabledError` is only thrown in the new version of the car, and not the old one. As a result, the driver using the car was not aware it and as a result the new car can't be replaced by the old car without changing the code using it. 

#### Liskov substitution principle - properties rule

In the two scenarios above, we have violated part two of the Liskov substitution principle, the **properties rule** it states:

1. invariants should be enforced in the subclass
2. evolutionary properties - the side effects of the subclass must be a subset of the superclass

During the 2010s came the hybrid era. The hybrid era meant incorporating it into the car.  As a result of incorporating the battery into the car, we had to make a few amendments, namely:

1. The maximum amount of logs that can be stored is 5
2. The check that ensured the telemetry was enabled when the engine was started was removed
3. The amount of fuel given being used by the car must be in multiples of 5

In [3]:
from app.battery import BatteryFactory
from app.f1_cars.hybrid_f1_car_with_telemetry_v1 import HybridF1CarWithTelemetry_v1

Alright now that we have the new car let's test it out!

In [4]:
f1_car = HybridF1CarWithTelemetry_v1(
    EngineFactory.create_engine_2010s_1_6_L_turbocharged_v1(),
    ChassisFactory.create_chassis_v1(),
    WheelsFactory.create_wheels_v1(),
    FuelTankFactory.create_fuel_tank_v1(),
    TelemetrySystemFactory.create_telemetry_system_v2(),
    BatteryFactory.create_battery_v1(),
)

NameError: name 'EngineFactory' is not defined

Because of the addition of the battery, the driver needs to be able to activate it and to be able to handle the error. Here the **signature rule** of the Liskov substituion principle described above is violated again. As a result, the `Driver` code implementation needs to be opened up again. Later we will see how the segregation of interfaces will resolve this issue

In [ ]:
from app.drivers.driver_v4 import Driver_v4

In [ ]:
driver = Driver_v4()
driver.start_car(f1_car)

Sweet now let's take it for a test drive!

In [ ]:
driver.enable_boost(f1_car)
driver.accelerate_car(f1_car, fuel_amount=10)

Try the boost

In [ ]:
driver.enable_boost(f1_car)
driver.accelerate_car(f1_car, fuel_amount=10)

Sweet now let's check the telemetry!

In [ ]:
telemetry = f1_car.get_telemetry_logs()

In [ ]:
telemetry

Why is this an empty list?! It is because an invariant enforced by previous versions of the car was removed and the driver forgot to enable the telemetry via `enable_car_telemetry`. In the past, the driver was notified of the `TelemetryNotEnabledError` and was able to rectify it by enabling the telemetry, but as a result of the invariant being removed the telemetry data isn't being recorded! To fix this, the invariant in the previous versions can be reintroduced:  

In [5]:
from app.f1_cars.hybrid_f1_car_with_telemetry_v2 import HybridF1CarWithTelemetry_v2

In [6]:
f1_car = HybridF1CarWithTelemetry_v2(
    EngineFactory.create_engine_2010s_1_6_L_turbocharged_v1(),
    ChassisFactory.create_chassis_v1(),
    WheelsFactory.create_wheels_v1(),
    FuelTankFactory.create_fuel_tank_v1(),
    TelemetrySystemFactory.create_telemetry_system_v2(),
    BatteryFactory.create_battery_v1(),
)
driver = Driver_v4()
driver.start_car(f1_car)

NameError: name 'EngineFactory' is not defined

Alright now to take it on a test drive!

In [50]:
for _ in range(10):
    driver.accelerate_car(f1_car, fuel_amount=5)

inject air into engine with better efficiency
inject fuel into engine with better efficiency
inject air into engine with better efficiency
inject fuel into engine with better efficiency
inject air into engine with better efficiency
inject fuel into engine with better efficiency
inject air into engine with better efficiency
inject fuel into engine with better efficiency
inject air into engine with better efficiency
inject fuel into engine with better efficiency
inject air into engine with better efficiency
inject fuel into engine with better efficiency
inject air into engine with better efficiency
inject fuel into engine with better efficiency
inject air into engine with better efficiency
inject fuel into engine with better efficiency
inject air into engine with better efficiency
inject fuel into engine with better efficiency
inject air into engine with better efficiency
inject fuel into engine with better efficiency


The introduction of the maximum number of logs constraint means only the last 5 logs can be recorded

In [51]:
f1_car.get_telemetry_logs()

[{'electricity': 45},
 {'electricity': 5},
 {'electricity': 10},
 {'electricity': 15},
 {'electricity': 20}]

In [52]:
def calculate_average_electricity_usage_for_each_acceleration(telemetry_logs):
    return [telemetry_log["electricity"] for telemetry_log in telemetry_logs] / len(
        telemetry_logs
    )

The function above requires all logs be recorded to see the fuel usage over the duration of the test, but only the last 5 logs are recorded so as a result the out of this function will be erroneous

In the two scenarios above, we have violated part two of the Liskov substitution principle, the **properties rule** it states:

1. invariants should be enforced in the subclass, telemetry should always be enabled when the engine is started
2. evolutionary properties - the side effects of the subclass must be a subset of the superclass. In our case, the superclass keeps a transaction of all the logs when the accelerator was pushed; whilst the subclass only keeps the latest x)

#### Liskov substitution principle - methods rule

The **methods rule** states:
1. Methods must have weaker preconditions
2. Methods must have stronger postconditions

For a given condition `C` a weaker condition `C_{weak}` can be thought of as a superset of the all the values that `C` can take. Equivalently, a stronger condition `C_{strong}` can be thought of as a subset of all the values that `C` can take

In [9]:
driver.accelerate_car(f1_car, fuel_amount=8)

NameError: name 'driver' is not defined

Here we can tell the driver to handle this but again, the driver code is opened up for modification

In [10]:
from app.drivers.driver_v5 import Driver_v5

In [11]:
f1_car = HybridF1CarWithTelemetry_v2(
    EngineFactory.create_engine_2010s_1_6_L_turbocharged_v1(),
    ChassisFactory.create_chassis_v1(),
    WheelsFactory.create_wheels_v1(),
    FuelTankFactory.create_fuel_tank_v1(),
    TelemetrySystemFactory.create_telemetry_system_v2(),
    BatteryFactory.create_battery_v1(),
)
driver = Driver_v5()
driver.accelerate_car(f1_car, fuel_amount=8)

NameError: name 'EngineFactory' is not defined

But stronger postconditions (initially the telemetry returns fuel & tire temp, but the subclass returns, fuel & electricity, hence telemetry analysis code downstream will break)

In [12]:
def calculate_average_fuel_usage_for_each_acceleration(telemetry_logs):
    return [telemetry_log["fuel"] for telemetry_log in telemetry_logs] / len(
        telemetry_logs
    )

In [57]:
telemetry = f1_car.get_current_telemetry()

In [58]:
calculate_average_fuel_usage_for_each_acceleration(telemetry)

KeyError: 'fuel'

Here, we can again modify the car to ensure the post conditions:

In [59]:
from app.f1_cars.hybrid_f1_car_with_telemetry_v3 import HybridF1CarWithTelemetry_v3

In [60]:
driver = Driver_v5()
f1_car = HybridF1CarWithTelemetry_v3(
    EngineFactory.create_engine_2010s_1_6_L_turbocharged_v1(),
    ChassisFactory.create_chassis_v1(),
    WheelsFactory.create_wheels_v1(),
    FuelTankFactory.create_fuel_tank_v1(),
    TelemetrySystemFactory.create_telemetry_system_v2(),
    BatteryFactory.create_battery_v1(),
)
driver.start_car(f1_car)

engine has started with turbocharger


In [61]:
telemetry = f1_car.get_current_telemetry()
analyze_telemetry(telemetry)

logic to analyze 100 for fuel telemetry
logic to analyze 0 for electricity telemetry


The **methods rule** states:
1. Methods must have weaker preconditions, in our case, the `accelerate` method has a stronger precondition as initially any amount of fuel was accepted, but in v6 now it only accepts fuel that are in multiples of 5
2. Methods must have stronger postconditions, in our case, the `get_telemetry_logs` method has a weaker postcondition as initially the telemetry from both fuel and electricity were returned, whereas now only the telemtetry from the fuel is returned

For a given condition `C` a weaker condition `C_{weak}` can be thought of as a superset of the all the values that `C` can take. Equivalently, a stronger condition `C_{strong}` can be thought of as a subset of all the values that `C` can take

While, SRP, DI, and OCP deals with the extendability of the code itself. The Liskov Substitution Principle deals with the clients of code, we see here that the car, is working as intended, but the clients of the car break as it doesn't adhere to the rules set forth by the Liskov Substitution Princple.

### Interface segregation principle

Although we've updated the car such that it adheres to there are some issues that still persist due to the change in the architecture of the car. For example, the introduction of the battery and a new error. As a result, the driver code needed to be opened up to deal with this error. To resolve so that the driver code doesn't need to be opened up for modification we will need to know the type of car we are dealing with then we will be able to create the correct kind of driver. 

So even though the driver depends on an abstraction, the `F1CarInterface` and not a concrete implementation, the abstraction is very broad and doesn't have enough information. This manifests itself in the `Driver` code where the `start_car` and `accelerate_car` has a bunch of code dealing with whether the car is a hybrid or non-hybrid car. The issue right now we don't know the type of `F1Car` we are dealing with, as a result we can only be defensive about our programming and catch errors then input the required logic (for example the `InvalidFuelAmountError`) as a result the driver code is polluted with a lot of code which is only specific to a type of car.

Moreover, the `F1CarInterface` doesn't work for the previous cars as we had to keep updating it, for example if we tried to use the `F1CarInterface` on `F1Car_v1` it wouldn't work, and if we had to make it work stubs would have to be created for all the methods that aren't currently implemented

To solve these issues, we can segregate the `F1CarInterface` into multiple different interfaces

In [62]:
from app.f1_cars.interface import (
    HybridAccelerableInterface,
    NonHybridAccelerableInterface,
    StartableInterface,
    TelemetryInterface,
)

In [63]:
from app.f1_cars.f1_car_v5 import F1Car_v5
from app.f1_cars.f1_car_with_telemetry_v4 import F1CarWithTelemetry_v4
from app.f1_cars.hybrid_f1_car_with_telemetry_v4 import HybridF1CarWithTelemetry_v4

In [68]:
hybrid_f1_car_with_telemetry = HybridF1CarWithTelemetry_v4(
    EngineFactory.create_engine_2010s_1_6_L_turbocharged_v1(),
    ChassisFactory.create_chassis_v1(),
    WheelsFactory.create_wheels_v1(),
    FuelTankFactory.create_fuel_tank_v1(),
    TelemetrySystemFactory.create_telemetry_system_v2(),
    BatteryFactory.create_battery_v1(),
)

In [69]:
non_hybrid_f1_car_with_telemetry = F1CarWithTelemetry_v4(
    EngineFactory.create_engine_1980s_1_5L_turbocharged_v3(),
    ChassisFactory.create_chassis_v1(),
    WheelsFactory.create_wheels_v1(),
    FuelTankFactory.create_fuel_tank_v1(),
    TelemetrySystemFactory.create_telemetry_system_v2(),
)

In [70]:
non_hybrid_f1_car = F1Car_v5(
    EngineFactory.create_engine_1950s_2_5L_naturally_aspirated_v2(),
    ChassisFactory.create_chassis_v1(),
    WheelsFactory.create_wheels_v1(),
    FuelTankFactory.create_fuel_tank_v1(),
)

Also, now there is more information about the characteristics of the car, therefore different driver classes can be created for different kinds of cars. The logic for determining what kind of driver to create can be encapsulated into a factory like so:

In [83]:
from app.drivers import DriverFactory

In [84]:
driver = DriverFactory.create_driver_from_car(hybrid_f1_car_with_telemetry)

In [85]:
type(driver)

app.drivers.driver_for_hybrid_with_telemetry.DriverForHybridWithTelemetry

In [86]:
driver = DriverFactory.create_driver_from_car(non_hybrid_f1_car_with_telemetry)

In [87]:
type(driver)

app.drivers.driver_for_non_hybrid_with_telemetry.DriverForNonHybridWithTelemetry

In [88]:
driver = DriverFactory.create_driver_from_car(non_hybrid_f1_car)

In [89]:
type(driver)

app.drivers.driver_for_non_hybrid_without_telemetry.DriverForNonHybridWithoutTelemetry

Moreover, if a new type of car needed to be modelled which has functions which can't be represented by the interface, then no code would have to be modified, we could simply add another if statement to the driver factory, add another interface and then you will be done!